# Granite-IO Interactive Chat Agent with vLLM

Welcome to this hands-on tutorial on **Granite-IO**, a powerful Python framework that enables you to transform how users interact with IBM Granite models and customize the input/output processing.

## What is Granite-IO?

Granite-IO is a framework that lets you:
- 🔄 Transform how users call or infer IBM Granite models
- 🎯 Customize how outputs are returned to users
- ⚡ Extend model functionality through input/output processing
- 🧠 Enable advanced features like "thinking" mode
- 📚 Build RAG (Retrieval-Augmented Generation) applications

## What We'll Build

In this notebook, we'll create an interactive chat agent that:
1. Uses **vLLM** as the backend for high-performance inference
2. Provides a simple chat interface where you can ask questions
3. Demonstrates both regular and "thinking" modes
4. Shows how easy it is to get started with Granite-IO
5. Uses the latest **Granite 3.3** models with enhanced capabilities

## Why Granite 3.3?

Granite 3.3 models offer significant improvements over previous versions:
- **🎯 Better Performance**: Enhanced reasoning and instruction-following
- **🧠 Advanced Thinking**: Improved structured reasoning with `<think>` tags
- **📈 Benchmark Gains**: Better scores on AlpacaEval-2.0 and Arena-Hard
- **🔧 More Capabilities**: Enhanced coding, mathematics, and multilingual support
- **📝 Fill-in-the-Middle**: Support for code completion tasks

## vLLM vs Ollama

While both can serve Granite models, they have different strengths:
- **vLLM**: High-performance inference server, great for production and batched requests
- **Ollama**: Easy local setup, great for development and single-user scenarios

## Prerequisites

- Python 3.10+
- vLLM installed and a Granite model available
- Or access to a vLLM server running Granite models

## Step 1: Installation and Setup

First, let's install the required packages. vLLM provides high-performance inference for transformer models.

In [ ]:
# Install granite-io with OpenAI compatibility and vLLM
!pip install granite-io[openai] vllm -q

print("✅ Granite-IO and vLLM installed successfully!")
print("📦 Next steps:")
print("   1. Choose a Granite model to run")
print("   2. Start vLLM server with the model")
print("   3. Connect Granite-IO to the vLLM server")
print()
print("💡 Common Granite models:")
print("   - ibm-granite/granite-3.3-8b-instruct")
print("   - ibm-granite/granite-3.3-2b-instruct")
print("   - ibm-granite/granite-3.3-3b-instruct")

## Step 2: Start vLLM Server

There are two ways to proceed:

### Option A: Start vLLM Server in Background (Recommended)
Run this command in a terminal to start vLLM with a Granite model:

```bash
vllm serve ibm-granite/granite-3.3-8b-instruct \
  --host 0.0.0.0 \
  --port 8000 \
  --served-model-name granite-3.3-8b
```

### Option B: Start vLLM Server from Notebook (Alternative)
Run the cell below to start vLLM in the background from this notebook.

In [ ]:
# Import required libraries
import requests
import json
import subprocess
import time
from granite_io import make_backend, make_io_processor
from granite_io.types import ChatCompletionInputs, UserMessage

# Configuration
VLLM_HOST = "localhost"
VLLM_PORT = "8000"
MODEL_NAME = "granite-3.3-8b"  # This will be the served model name
HUGGINGFACE_MODEL = "ibm-granite/granite-3.3-8b-instruct"

def check_vllm_server():
    """Check if vLLM server is running and accessible"""
    try:
        response = requests.get(f"http://{VLLM_HOST}:{VLLM_PORT}/v1/models", timeout=5)
        if response.status_code == 200:
            models = response.json()
            print("✅ vLLM server is running and accessible!")
            print("📋 Available models:")
            for model in models.get("data", []):
                print(f"   - {model['id']}")
            return True
        else:
            print("❌ vLLM server responded but with error")
            return False
    except requests.exceptions.RequestException as e:
        print(f"❌ Cannot connect to vLLM server: {e}")
        return False

def start_vllm_server():
    """Start vLLM server in background (Option B)"""
    print("🚀 Starting vLLM server in background...")
    print(f"📦 Model: {HUGGINGFACE_MODEL}")
    print(f"🌐 Server: http://{VLLM_HOST}:{VLLM_PORT}")
    print("⏳ This may take a few minutes for first-time model download...")
    
    # Start vLLM server
    cmd = [
        "vllm", "serve", HUGGINGFACE_MODEL,
        "--host", "0.0.0.0",
        "--port", VLLM_PORT,
        "--served-model-name", MODEL_NAME
    ]
    
    try:
        # Start the process in background
        process = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        print(f"📋 vLLM server started with PID: {process.pid}")
        print("⏳ Waiting for server to be ready...")
        
        # Wait for server to be ready (up to 60 seconds)
        for i in range(12):  # 12 * 5 = 60 seconds
            time.sleep(5)
            if check_vllm_server():
                return True
            print(f"   Still waiting... ({(i+1)*5}s)")
        
        print("⚠️  Server might still be starting. Try checking manually.")
        return False
        
    except Exception as e:
        print(f"❌ Failed to start vLLM server: {e}")
        return False

# Check if server is already running, otherwise offer to start it
if not check_vllm_server():
    print("\n💡 vLLM server not found. Options:")
    print("   1. Start it manually in terminal (recommended)")
    print("   2. Uncomment the line below to start from notebook")
    print()
    # Uncomment the next line to start vLLM server from notebook:
    # start_vllm_server()

## Step 3: Initialize Granite-IO with vLLM Backend

Now let's create our Granite-IO processor that will interface with the vLLM server. The setup is very similar to Ollama since vLLM also provides an OpenAI-compatible API!

In [ ]:
# Create the backend configuration for vLLM
# vLLM uses OpenAI-compatible API, so we use the "openai" backend
backend_config = {
    "model_name": MODEL_NAME,  # The served model name in vLLM
    "base_url": f"http://{VLLM_HOST}:{VLLM_PORT}/v1",  # vLLM's OpenAI-compatible endpoint
    "api_key": "vllm"  # vLLM doesn't require a real API key, but the client expects one
}

try:
    # Create the backend and IO processor
    backend = make_backend("openai", backend_config)
    io_processor = make_io_processor(MODEL_NAME, backend=backend)
    
    print("🎉 Granite-IO processor created successfully!")
    print(f"🤖 Using model: {MODEL_NAME}")
    print("🔌 Backend: vLLM (via OpenAI-compatible API)")
    print(f"🌐 Server: {backend_config['base_url']}")
    
    # Test basic connectivity
    print("\n🔍 Testing connection...")
    test_response = requests.get(f"{backend_config['base_url'].replace('/v1', '')}/health")
    if test_response.status_code == 200:
        print("✅ vLLM server is healthy and ready!")
    else:
        print("⚠️  Server responded but health check unclear")
    
except Exception as e:
    print(f"❌ Error creating processor: {e}")
    print("💡 Make sure vLLM server is running and the model is loaded")
    print(f"   Try: curl {backend_config['base_url']}/models")

## Step 4: Your First Chat with Granite-IO + vLLM

Let's test our setup with a simple question!

In [ ]:
# Let's try a simple question and measure performance
import time

user_question = "What is the capital of France and why is it important?"

# Create a user message
messages = [UserMessage(content=user_question)]

# Create chat completion input
chat_input = ChatCompletionInputs(messages=messages)

try:
    print(f"🗣️  You: {user_question}")
    print("🤖 Granite (via vLLM): ", end="")
    
    # Measure response time
    start_time = time.time()
    outputs = io_processor.create_chat_completion(chat_input)
    end_time = time.time()
    
    response = outputs.results[0].next_message.content
    response_time = end_time - start_time
    
    print(response)
    print(f"\n⚡ Response time: {response_time:.2f} seconds")
    print("✅ Success! Your Granite-IO + vLLM chat agent is working!")
    
except Exception as e:
    print(f"❌ Error: {e}")
    print("💡 Check that vLLM server is running and the model is loaded")
    print(f"   Debug: curl {backend_config['base_url']}/models")

## Step 5: Exploring the "Thinking" Feature

One of Granite-IO's powerful features is the ability to see the model's reasoning process. Let's try this with vLLM's high-performance backend!

In [ ]:
# Let's ask a complex question that benefits from reasoning
thinking_question = "How would you design a scalable microservices architecture for an e-commerce platform?"

# Create messages for thinking mode
messages = [UserMessage(content=thinking_question)]

try:
    print(f"🗣️  You: {thinking_question}")
    print("\n" + "="*70)
    
    # First, let's see the response WITHOUT thinking
    print("🤖 WITHOUT THINKING:")
    print("-" * 30)
    start_time = time.time()
    regular_outputs = io_processor.create_chat_completion(ChatCompletionInputs(messages=messages))
    regular_time = time.time() - start_time
    
    regular_response = regular_outputs.results[0].next_message.content
    print(regular_response)
    print(f"⚡ Time: {regular_time:.2f}s")
    
    print("\n" + "="*70)
    
    # Now WITH thinking enabled
    print("🧠 WITH THINKING:")
    print("-" * 30)
    start_time = time.time()
    thinking_outputs = io_processor.create_chat_completion(
        ChatCompletionInputs(messages=messages, thinking=True)
    )
    thinking_time = time.time() - start_time
    
    # Show the reasoning process
    print("💭 Model's Thoughts:")
    reasoning = thinking_outputs.results[0].next_message.reasoning_content
    if reasoning:
        print(reasoning)
    else:
        print("(No reasoning content available)")
    
    print("\n🎯 Final Response:")
    thinking_response = thinking_outputs.results[0].next_message.content
    print(thinking_response)
    print(f"⚡ Time: {thinking_time:.2f}s")
    
    print(f"\n📊 Performance comparison:")
    print(f"   Regular mode: {regular_time:.2f}s")
    print(f"   Thinking mode: {thinking_time:.2f}s")
    print(f"   vLLM advantage: High throughput and efficient GPU utilization!")
    
except Exception as e:
    print(f"❌ Error: {e}")

## Step 6: Interactive Chat Agent with vLLM Performance

Now let's create a full interactive chat experience powered by vLLM's high-performance inference! This is the same chat interface as the Ollama notebook, showcasing how Granite-IO provides a consistent API across different backends.

In [ ]:
def interactive_chat_vllm():
    """
    Interactive chat function with conversation history and performance monitoring
    """
    print("🎯 Interactive Granite Chat Agent (vLLM Backend)")
    print("=" * 60)
    print("💬 Type your questions (or 'quit' to exit)")
    print("🧠 Add 'think:' at the start to enable thinking mode")
    print("🔄 Type 'clear' to clear conversation history")
    print("📊 Type 'stats' to see performance statistics")
    print("=" * 60)
    
    # Conversation history and stats
    conversation_history = []
    response_times = []
    
    while True:
        try:
            # Get user input
            user_input = input("\n🗣️  You: ").strip()
            
            if user_input.lower() == 'quit':
                print("👋 Goodbye! Thanks for chatting!")
                break
            elif user_input.lower() == 'clear':
                conversation_history = []
                response_times = []
                print("🧹 Conversation history cleared!")
                continue
            elif user_input.lower() == 'stats':
                if response_times:
                    avg_time = sum(response_times) / len(response_times)
                    min_time = min(response_times)
                    max_time = max(response_times)
                    print(f"📊 Performance Stats:")
                    print(f"   Total responses: {len(response_times)}")
                    print(f"   Average time: {avg_time:.2f}s")
                    print(f"   Fastest: {min_time:.2f}s")
                    print(f"   Slowest: {max_time:.2f}s")
                    print(f"   Backend: vLLM (optimized for throughput)")
                else:
                    print("📊 No performance data yet. Ask some questions first!")
                continue
            elif not user_input:
                continue
            
            # Check if thinking mode is requested
            thinking_mode = user_input.lower().startswith('think:')
            if thinking_mode:
                user_input = user_input[6:].strip()  # Remove 'think:' prefix
            
            # Add user message to history
            conversation_history.append(UserMessage(content=user_input))
            
            # Create chat input with full conversation history
            chat_input = ChatCompletionInputs(
                messages=conversation_history,
                thinking=thinking_mode
            )
            
            # Get response with timing
            print("🤖 Granite (vLLM): ", end="")
            if thinking_mode:
                print("(thinking...)")
            
            start_time = time.time()
            outputs = io_processor.create_chat_completion(chat_input)
            end_time = time.time()
            response_time = end_time - start_time
            response_times.append(response_time)
            
            result = outputs.results[0].next_message
            
            # Show thinking if available
            if thinking_mode and result.reasoning_content:
                print(f"💭 Thoughts: {result.reasoning_content}")
                print(f"🎯 Response: {result.content}")
            else:
                print(result.content)
            
            print(f"⚡ ({response_time:.2f}s)")
            
            # Add assistant response to history (simplified - just the content)
            from granite_io.types import AssistantMessage
            conversation_history.append(AssistantMessage(content=result.content))
            
        except KeyboardInterrupt:
            print("\n👋 Chat interrupted. Goodbye!")
            break
        except Exception as e:
            print(f"\n❌ Error: {e}")
            print("💡 Try again or check your vLLM server")

# Start the interactive chat
interactive_chat_vllm()

## 🎉 Congratulations!

You've successfully created an interactive chat agent using **Granite-IO** with **vLLM**! 

### What You've Learned:

1. **🔧 Setup**: How to install and configure Granite-IO with vLLM
2. **🚀 Performance**: vLLM's high-performance inference capabilities  
3. **🔌 Backend**: Using vLLM as a scalable backend for Granite models
4. **💬 Chat**: Creating chat completions with performance monitoring
5. **🧠 Thinking Mode**: Accessing the model's reasoning process
6. **🔄 Interactive Chat**: Building a conversational agent with statistics

### vLLM Advantages:

- **⚡ High Throughput**: Optimized for serving multiple requests efficiently
- **🎯 Low Latency**: Fast inference with advanced batching and caching
- **📈 Scalability**: Production-ready with horizontal scaling capabilities
- **🔧 Flexibility**: Supports various model architectures and quantization
- **💾 Memory Efficiency**: Advanced memory management and optimization

### Granite-IO + vLLM vs Granite-IO + Ollama:

| Feature | vLLM | Ollama |
|---------|------|--------|
| **Performance** | High throughput, optimized for production | Good for development and single-user |
| **Setup** | Requires more configuration | Simple, one-command setup |
| **Use Case** | Production, multiple users, high load | Development, personal use, demos |
| **Resource Usage** | More memory efficient under load | Simpler resource management |
| **API Compatibility** | OpenAI-compatible | OpenAI-compatible |

### Key Granite-IO Concepts (Same Across Backends!):

- **`make_backend()`**: Creates the connection to your model backend
- **`make_io_processor()`**: The main interface for processing requests
- **`ChatCompletionInputs`**: Structure your chat inputs uniformly
- **`thinking=True`**: Enable reasoning mode across any backend
- **Consistent API**: Same code works with Ollama, vLLM, or cloud providers

### Next Steps:

- Try batch processing multiple requests with vLLM
- Experiment with different Granite model sizes
- Explore vLLM's advanced features (quantization, tensor parallelism)
- Build production applications with load balancing
- Implement custom Granite-IO processors for specialized use cases

### 🔗 Learn More:

- [Granite-IO Documentation](https://github.com/ibm-granite/granite-io)
- [vLLM Documentation](https://docs.vllm.ai/)
- [Granite Models on Hugging Face](https://huggingface.co/ibm-granite)
- [vLLM Performance Tuning Guide](https://docs.vllm.ai/en/latest/)